In [ ]:
from src.data.data import *
from src.embedor import *
from src.plotting import *
import matplotlib
import seaborn as sns
import umap
import numpy as np
from sklearn.manifold import TSNE, Isomap, SpectralEmbedding
import phate

%load_ext autoreload

In [310]:

def get_low_energy_graph(embedor_object: EmbedOR, edge_pctile=33):
    """
    Returns a graph with edges that are below the specified percentile of edge distances.
    """
    G_low_energy = embedor_object.G.copy()
    edge_dists = {
        (u, v): embedor_object.G[u][v]['energy'] for u, v in embedor_object.G.edges
    }
    edge_distances = np.array(list(edge_dists.values()))
    for idx, (u, v) in enumerate(embedor_object.G.edges):
        if edge_distances[idx] > np.percentile(edge_distances, edge_pctile):
            G_low_energy.remove_edge(u, v)
    return G_low_energy

def eval_low_energy_edges(embedor_object: EmbedOR, edge_pctile=33, clusters=None):
    # get number of cluster bridging edges in the entire graph
    G_full = embedor_object.G.copy()
    n_brige_full = 0
    for edge in G_full.edges:
        u, v = edge
        if clusters[u] != clusters[v]:
            n_brige_full += 1
    # check what percent of low energy edges bridge the clusters
    G_low_energy = get_low_energy_graph(embedor_object, edge_pctile=edge_pctile)
    n_bridge_low_energy = 0
    for edge in G_low_energy.edges:
        u, v = edge
        if clusters[u] != clusters[v]:
            n_bridge_low_energy += 1
    return n_bridge_low_energy, n_brige_full


exp_params = {
    'p': 3,
}

In [ ]:
def circles_metric_eval(n_iter=10):
    n_points = 5000
    # concentric circles
    noise = 0.1
    noise_thresh = None
    n_bridge_full_array = []
    n_bridge_low_energy_array = []
    for iter in range(n_iter):
        print(f"Iteration {iter+1}/{n_iter}")
        return_dict = concentric_circles(n_points=n_points, factor=0.4, noise=noise, noise_thresh=noise_thresh)
        embedor = EmbedOR(exp_params)
        embedding = embedor.fit_transform(return_dict['data'])
        n_bridge_low_energy, n_brige_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=return_dict['cluster'])
        n_bridge_low_energy_array.append(n_bridge_low_energy)
        n_bridge_full_array.append(n_brige_full)

    n_bridge_full_array = np.array(n_bridge_full_array)
    n_bridge_low_energy_array = np.array(n_bridge_low_energy_array)
    return n_bridge_low_energy_array, n_bridge_full_array

circles_n_bridge_low_energy, circles_n_bridge_full = circles_metric_eval(n_iter=10)
print(f'(mean, std) #  bridging edges in low energy graph: {np.mean(circles_n_bridge_low_energy):.2f} ± {np.std(circles_n_bridge_low_energy):.2f}')
print(f'(mean, std) #  bridging edges in full graph: {np.mean(circles_n_bridge_full):.2f} ± {np.std(circles_n_bridge_full):.2f}')

Iteration 1/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 2/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 3/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 4/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 5/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 6/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochas

In [312]:

def moons_metric_eval(n_iter=10):
    n_points = 3000
    noise = 0.125
    noise_thresh = None
    n_bridge_full_array = []
    n_bridge_low_energy_array = []
    for iter in range(n_iter):
        print(f"Iteration {iter+1}/{n_iter}")
        return_dict = moons(n_points=n_points, noise=noise, noise_thresh=noise_thresh)
        embedor = EmbedOR(exp_params)
        embedding = embedor.fit_transform(return_dict['data'])
        n_bridge_low_energy, n_brige_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=return_dict['cluster'])
        n_bridge_low_energy_array.append(n_bridge_low_energy)
        n_bridge_full_array.append(n_brige_full)
    n_bridge_full_array = np.array(n_bridge_full_array)
    n_bridge_low_energy_array = np.array(n_bridge_low_energy_array)
    return n_bridge_low_energy_array, n_bridge_full_array

moons_n_bridge_low_energy, moons_n_bridge_full = moons_metric_eval(n_iter=10)
print(f'(mean, std) #  bridging edges in low energy graph: {np.mean(moons_n_bridge_low_energy):.2f} ± {np.std(moons_n_bridge_low_energy):.2f}')
print(f'(mean, std) #  bridging edges in full graph: {np.mean(moons_n_bridge_full):.2f} ± {np.std(moons_n_bridge_full):.2f}')   


Iteration 1/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 2/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 3/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 4/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 5/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 6/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochas

In [313]:
def torus_metric_eval(n_iter=10):
    n_points = 5000
    noise = 0.5
    noise_thresh = None
    n_bridge_full_array = []
    n_bridge_low_energy_array = []
    for iter in range(n_iter):
        print(f"Iteration {iter+1}/{n_iter}")
        return_dict = torus(n_points=n_points, noise=noise, noise_thresh=noise_thresh, supersample=False, double=True)
        embedor = EmbedOR(exp_params)
        embedding = embedor.fit_transform(return_dict['data'])
        n_bridge_low_energy, n_brige_full = eval_low_energy_edges(embedor, edge_pctile=33, clusters=return_dict['cluster'])
        n_bridge_low_energy_array.append(n_bridge_low_energy)
        n_bridge_full_array.append(n_brige_full)
    n_bridge_full_array = np.array(n_bridge_full_array)
    n_bridge_low_energy_array = np.array(n_bridge_low_energy_array)
    return n_bridge_low_energy_array, n_bridge_full_array

torus_n_bridge_low_energy, torus_n_bridge_full = torus_metric_eval(n_iter=10)
print(f'(mean, std) #  bridging edges in low energy graph: {np.mean(torus_n_bridge_low_energy):.2f} ± {np.std(torus_n_bridge_low_energy):.2f}')
print(f'(mean, std) #  bridging edges in full graph: {np.mean(torus_n_bridge_full):.2f} ± {np.std(torus_n_bridge_full):.2f}')

Iteration 1/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 2/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 3/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 4/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 5/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochastic Neighbor Embedding...
Iteration 6/10
Building nearest neighbor graph...
Computing distances...
Computing affinities...
Updating the graph attributes...
Running Stochas